# 후보 수와 리랭커 두 모델 비교

이 노트북은 공개 PDF 6건과 Ch11의 30문항을 사용합니다. 첫 셀부터 순서대로 실행합니다.


## 입력과 모델 준비

동일한 30문항과 후보 순서를 유지하고 CrossEncoder 모델과 후보 수만 바꿉니다.


In [1]:
from pathlib import Path
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "labs/part3/ch12-retrieval-tuning").exists())
data = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-documents.json").read_text())
queries = json.loads((root / "labs/part3/ch12-retrieval-tuning/data/advanced-queries.json").read_text())
chunks, parents = data["chunks"], {item["doc_id"]: item for item in data["parents"]}
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)
matrix = vectorizer.fit_transform([item["text"] for item in chunks])
print(f"부모 문서 {len(parents)}건 · 자식 청크 {len(chunks)}개 · 평가 질문 {len(queries)}개")

부모 문서 6건 · 자식 청크 175개 · 평가 질문 30개


In [2]:
from sentence_transformers import CrossEncoder
from time import perf_counter

model_names = ["BAAI/bge-reranker-v2-m3", "Dongjin-kr/ko-reranker"]
models = {name: CrossEncoder(name, max_length=256, device="cpu") for name in model_names}
print("모델 준비:", list(models))

[사용자 폴더]/working/claude/projects/한국경제신문/labs/part3/ch12-retrieval-tuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 24080.90it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7792.20it/s]

모델 준비: ['BAAI/bge-reranker-v2-m3', 'Dongjin-kr/ko-reranker']


## 평가 함수

고정 후보에서 고유 정답 부모 문서 중 검색된 비율과 상위 5개 순위의 nDCG@5, 실제 추론 시간을 기록합니다.


In [3]:
def eligible_items(subset):
    return [item for item in subset if item.get("answerable", True) and item["gold_doc_ids"]]

raw_dev_queries = [item for item in queries if item["split"] == "dev"]
dev_queries = eligible_items(raw_dev_queries)
excluded_query_count = len(raw_dev_queries) - len(dev_queries)
print("개발 문항:", len(dev_queries), "· 검색 평균 제외:", excluded_query_count)

개발 문항: 15 · 검색 평균 제외: 0


In [4]:
candidate_sets = {}
for item in dev_queries:
    scores = cosine_similarity(vectorizer.transform([item["question"]]), matrix)[0]
    candidate_sets[item["qid"]] = np.argsort(-scores)[:20]
def score_fixed_candidates(model):
    started, scored = perf_counter(), {}
    for item in dev_queries:
        idxs = candidate_sets[item["qid"]]
        pairs = [(item["question"], chunks[i]["text"]) for i in idxs]
        scored[item["qid"]] = model.predict(pairs, show_progress_bar=False)
    return scored, perf_counter() - started
def evaluate(scored, candidate_k):
    if not dev_queries:
        raise ValueError("평가 가능한 개발 문항이 없습니다")
    recalls, ndcgs = [], []
    for item in dev_queries:
        idxs = candidate_sets[item["qid"]][:candidate_k]
        order = np.argsort(-scored[item["qid"]][:candidate_k])[:5]
        top_ids = list(dict.fromkeys(chunks[idxs[pos]]["parent_id"] for pos in order))
        gold = set(item["gold_doc_ids"])
        recalls.append(len(gold & {chunks[i]["parent_id"] for i in idxs}) / len(gold))
        rank = next((i for i, doc_id in enumerate(top_ids, 1) if doc_id in gold), None)
        ndcgs.append(0.0 if rank is None else 1 / np.log2(rank + 1))
    return float(np.mean(recalls)), float(np.mean(ndcgs))

## 두 모델과 세 후보 수 비교

두 모델을 동일한 상위 20개 후보에 한 번씩 실행하고, 저장한 점수로 후보 수 5·10·20의 지표 6행을 계산합니다. 시간은 후보 수별 시간이 아니라 모델별 전체 점수화 시간입니다. 결과는 이 컴퓨터에서 관찰한 값이며 다른 장치에서는 시간이 달라집니다.


In [5]:
rows = []
for name, model in models.items():
    scored, seconds = score_fixed_candidates(model)
    for candidate_k in (5, 10, 20):
        recall, ndcg = evaluate(scored, candidate_k)
        rows.append((name.split("/")[-1], candidate_k, recall, ndcg, seconds))
for row in rows:
    print(f"{row[0]:24s} k={row[1]:2d} recall={row[2]:.3f} nDCG={row[3]:.3f} time={row[4]:.2f}s")

bge-reranker-v2-m3       k= 5 recall=1.000 nDCG=1.000 time=15.82s
bge-reranker-v2-m3       k=10 recall=1.000 nDCG=1.000 time=15.82s
bge-reranker-v2-m3       k=20 recall=1.000 nDCG=1.000 time=15.82s
ko-reranker              k= 5 recall=1.000 nDCG=1.000 time=19.56s
ko-reranker              k=10 recall=1.000 nDCG=1.000 time=19.56s
ko-reranker              k=20 recall=1.000 nDCG=1.000 time=19.56s
